# SplitFed & AD-SFL Setup Demo

This notebook demonstrates how to use the dataset and model modules we've implemented. It covers loading the data, creating federated partitions and reference datasets, and initializing and testing the split models (Client and Server side).

In [2]:
# --- First cell: clone/update repo + install from inner pyproject (monorepo-style) ---

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/tomal66/ad-sfl-experiments.git"
REPO_DIR = Path("ad-sfl-experiments")
BRANCH = "main"

# Inner project directory that contains pyproject.toml
PROJECT_DIR = REPO_DIR / "ad-sfl-experiments"

def run(cmd, cwd=None):
    print(f"\n$ {' '.join(cmd)}" + (f"  (cwd={cwd})" if cwd else ""))
    subprocess.run(cmd, cwd=cwd, check=True)

# 1) Clone or update
if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"'{REPO_DIR}' exists but is not a git repo. Rename/delete it and rerun.")
    run(["git", "fetch", "origin"], cwd=str(REPO_DIR))
    run(["git", "checkout", BRANCH], cwd=str(REPO_DIR))
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=str(REPO_DIR))

# 2) Install from inner pyproject
pyproject = PROJECT_DIR / "pyproject.toml"
if not pyproject.exists():
    raise RuntimeError(
        f"pyproject.toml not found at {pyproject}. "
        "Double-check the inner project path."
    )

run(["python", "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
# Editable install so changes in the repo take effect immediately
run(["python", "-m", "pip", "install", "-e", str(PROJECT_DIR)])

print("\nRepo ready and installed from:", PROJECT_DIR.resolve())


$ git fetch origin  (cwd=ad-sfl-experiments)

$ git checkout main  (cwd=ad-sfl-experiments)

$ git pull --ff-only origin main  (cwd=ad-sfl-experiments)

$ python -m pip install -U pip setuptools wheel

$ python -m pip install -e ad-sfl-experiments\ad-sfl-experiments

Repo ready and installed from: C:\Users\BIM\ad-sfl-experiments\ad-sfl-experiments


In [5]:
import os
import sys
import torch

# Ensure the src module can be imported
sys.path.append(os.path.join(os.getcwd(), 'ad-sfl-experiments/ad-sfl-experiments'))

from src.data.datasets import get_dataset
from src.data.partition import partition_iid, partition_dirichlet
from src.data.reference import create_reference_dataset
from src.models.registry import get_client_model, get_server_model
from src.models.split import split_model_output

NameError: name 'torch' is not defined

## 1. Data Setup & Partitioning
Here we load the MNIST dataset, extract a reference dataset for AD-SFL, and partition the rest into IID subsets for federated clients.

In [ ]:
# Load full datasets
print("Loading MNIST dataset...")
train_ds, test_ds = get_dataset('mnist', data_dir='./data')
print(f"Original Train Size: {len(train_ds)} | Test Size: {len(test_ds)}")

# 1. Create Reference Dataset (Held by server for AD-SFL)
new_train_ds, reference_ds = create_reference_dataset(train_ds, reference_size=1000, stratify=True)
print(f"\nAfter Reference Extraction:")
print(f"New Train Size: {len(new_train_ds)} | Reference Size: {len(reference_ds)}")

# 2. Partition Data among Clients
num_clients = 5
client_partitions = partition_iid(new_train_ds, num_clients=num_clients)
print(f"\nIID Partitions for {num_clients} clients:")
for i, partition in enumerate(client_partitions):
    print(f"  Client {i+1} dataset size: {len(partition)}")

## 2. Models Setup (Split Architecture)
We instantiate the Client-side and Server-side architectures. For MNIST, we use the supported LeNet implementation.

In [ ]:
# Instantiate Split Models
print("Initializing Client and Server LeNet Models...")
client_model = get_client_model('lenet')
server_model = get_server_model('lenet', num_classes=10)

print("\nClient Model Architecture:")
print(client_model)

print("\nServer Model Architecture:")
print(server_model)

## 3. Dummy Forward Pass
Demonstrating the data flow from the client to the server during a forward pass.

In [ ]:
batch_size = 4
# Create a dummy tensor simulating a batch of MNIST images (B, C, H, W)
dummy_input = torch.randn(batch_size, 1, 28, 28) 

# --- CLIENT SIDE ---
# Client performs forward pass up to the split point
client_output = client_model(dummy_input)
print(f"Client Input Shape: {dummy_input.shape}")
print(f"Client Output Shape (Smashed Data): {client_output.shape}")

# The output is prepared for the server. 
# 'server_input' gets requires_grad=True so the server graph can differentiate it.
orig_out, server_input = split_model_output(client_output)

# --- COMMUNICATION LINK ---
# (In a real setup, server_input is serialized and sent over network)

# --- SERVER SIDE ---
# Server finishes the forward pass
server_output = server_model(server_input)
print(f"Server Final Predictions Shape: {server_output.shape}")

print("\nSplitFed forward pass completed successfully!")